# Factory Scheduling Demo Data Generator

This notebook builds a compact but realistic dataset for a **production scheduling / job-shop scheduling** demo, with special attention to **machine downtime with uncertain duration**.

The idea is to generate one shared dataset that can be reused across different approaches:
- a classical scheduler or rescheduler,
- a quantum or quantum-inspired optimizer,
- KPI comparisons on exactly the same input data.

## What this notebook exports

The notebook writes a small dataset bundle to disk as CSV files:

- `machines.csv` — machine master data,
- `shifts.csv` — available working windows,
- `orders.csv` — production orders,
- `operations.csv` — routing operations,
- `downtime_events.csv` — disruption events,
- `scenarios.csv` — scenario metadata.

In addition to the synthetic dataset, the notebook also creates a **small embedded benchmark instance** that is useful for debugging, regression checks, and quick solver smoke tests.

In [1]:
from __future__ import annotations

import math
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

OUTPUT_DIR = Path("generated_factory_demo_data")
OUTPUT_DIR.mkdir(exist_ok=True)

BASE_DATE = pd.Timestamp("2026-04-20 08:00:00")
TIME_UNIT_MIN = 5  # Keep everything on a 5-minute planning grid.

## 1. Data schema

The schema below is intentionally small, but still practical enough for scheduling demos and solver prototyping.

### `machines.csv`
- `machine_id`
- `machine_group`
- `machine_name`
- `daily_capacity_minutes`
- `efficiency`
- `can_preempt`

### `orders.csv`
- `order_id`
- `product_family`
- `priority`
- `release_time`
- `deadline`
- `customer_segment`

### `operations.csv`
- `operation_id`
- `order_id`
- `sequence_index`
- `machine_group_required`
- `preferred_machine_id`
- `processing_time_minutes`
- `setup_time_minutes`
- `release_time`
- `deadline`
- `is_outsourcable`

### `downtime_events.csv`
- `event_id`
- `machine_id`
- `event_start`
- `estimated_duration_minutes`
- `actual_duration_minutes`
- `reason`
- `scenario_name`

In [2]:
# Core reference data used throughout the notebook.
MACHINE_GROUPS = {
    "CUT": ["M_CUT_01", "M_CUT_02"],
    "WELD": ["M_WELD_01", "M_WELD_02"],
    "PAINT": ["M_PAINT_01"],
    "ASSY": ["M_ASSY_01", "M_ASSY_02"],
    "PACK": ["M_PACK_01"],
}

PRODUCT_ROUTINGS = {
    "FRAME_A": ["CUT", "WELD", "PAINT", "ASSY", "PACK"],
    "FRAME_B": ["CUT", "WELD", "ASSY", "PACK"],
    "KIT_C": ["CUT", "ASSY", "PACK"],
    "MODULE_D": ["WELD", "PAINT", "ASSY"],
}

CUSTOMER_SEGMENTS = ["OEM", "Distributor", "Rush", "Service"]
PRIORITY_WEIGHTS = {1: "critical", 2: "high", 3: "normal", 4: "low"}

In [3]:
def generate_machines(machine_groups: Dict[str, List[str]]) -> pd.DataFrame:
    """Create a simple machine master table."""
    rows = []
    for group, machines in machine_groups.items():
        for mid in machines:
            rows.append(
                {
                    "machine_id": mid,
                    "machine_group": group,
                    "machine_name": mid.replace("_", " "),
                    "daily_capacity_minutes": 9 * 60,
                    "efficiency": round(np.random.uniform(0.85, 1.05), 3),
                    "can_preempt": False,
                }
            )
    return pd.DataFrame(rows)


def generate_shifts(machines_df: pd.DataFrame, num_days: int = 3) -> pd.DataFrame:
    """Generate one working shift per machine per day."""
    rows = []
    for _, row in machines_df.iterrows():
        for day in range(num_days):
            start = BASE_DATE + pd.Timedelta(days=day)
            end = start + pd.Timedelta(hours=9)
            rows.append(
                {
                    "machine_id": row["machine_id"],
                    "shift_start": start,
                    "shift_end": end,
                    "is_working": True,
                }
            )
    return pd.DataFrame(rows)


def sample_processing_time(machine_group: str) -> int:
    """Sample a processing time and snap it to the common time grid."""
    ranges = {
        "CUT": (20, 90),
        "WELD": (30, 120),
        "PAINT": (40, 100),
        "ASSY": (30, 150),
        "PACK": (15, 60),
    }
    lo, hi = ranges[machine_group]
    duration = int(np.random.randint(lo // TIME_UNIT_MIN, hi // TIME_UNIT_MIN + 1) * TIME_UNIT_MIN)
    return max(duration, TIME_UNIT_MIN)


def sample_setup_time(machine_group: str) -> int:
    """Sample a setup time for the required machine group."""
    ranges = {
        "CUT": (5, 20),
        "WELD": (5, 25),
        "PAINT": (10, 30),
        "ASSY": (5, 20),
        "PACK": (0, 10),
    }
    lo, hi = ranges[machine_group]
    setup = int(np.random.randint(lo // TIME_UNIT_MIN, hi // TIME_UNIT_MIN + 1) * TIME_UNIT_MIN)
    return max(setup, 0)


def generate_orders_and_operations(
    num_orders: int,
    machine_groups: Dict[str, List[str]],
    product_routings: Dict[str, List[str]],
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Generate synthetic orders together with their routing operations."""
    order_rows = []
    op_rows = []

    for idx in range(1, num_orders + 1):
        order_id = f"ORD_{idx:03d}"
        product_family = random.choice(list(product_routings.keys()))
        routing = product_routings[product_family]
        priority = np.random.choice([1, 2, 3, 4], p=[0.1, 0.2, 0.5, 0.2])

        # Spread releases over the first 90 minutes to avoid a fully synchronized start.
        release_offset_slots = np.random.randint(0, 18)
        release_time = BASE_DATE + pd.Timedelta(minutes=int(release_offset_slots * TIME_UNIT_MIN))

        # First estimate the nominal workload, then build a deadline with extra slack.
        nominal_total = 0
        for g in routing:
            nominal_total += sample_processing_time(g) + sample_setup_time(g)

        slack_factor = np.random.uniform(1.2, 1.8)
        deadline = release_time + pd.Timedelta(
            minutes=int(nominal_total * slack_factor + np.random.randint(60, 240))
        )

        order_rows.append(
            {
                "order_id": order_id,
                "product_family": product_family,
                "priority": int(priority),
                "priority_label": PRIORITY_WEIGHTS[int(priority)],
                "release_time": release_time,
                "deadline": deadline,
                "customer_segment": random.choice(CUSTOMER_SEGMENTS),
            }
        )

        for seq, group in enumerate(routing, start=1):
            candidate_machines = machine_groups[group]
            preferred_machine_id = random.choice(candidate_machines)
            proc = sample_processing_time(group)
            setup = sample_setup_time(group)

            op_rows.append(
                {
                    "operation_id": f"{order_id}_OP_{seq:02d}",
                    "order_id": order_id,
                    "sequence_index": seq,
                    "machine_group_required": group,
                    "preferred_machine_id": preferred_machine_id,
                    "processing_time_minutes": proc,
                    "setup_time_minutes": setup,
                    "release_time": release_time,
                    "deadline": deadline,
                    "is_outsourcable": bool(np.random.rand() < 0.08),
                }
            )

    return pd.DataFrame(order_rows), pd.DataFrame(op_rows)

## 2. Synthetic dataset generation

This section creates the main synthetic dataset for the demo.

Typical knobs you may want to tune later:
- number of orders,
- planning horizon in days,
- routing mix,
- load density,
- deadline tightness.

In [4]:
machines_df = generate_machines(MACHINE_GROUPS)
shifts_df = generate_shifts(machines_df, num_days=3)
orders_df, operations_df = generate_orders_and_operations(
    num_orders=30,
    machine_groups=MACHINE_GROUPS,
    product_routings=PRODUCT_ROUTINGS,
)

print("machines:", machines_df.shape)
print("shifts:", shifts_df.shape)
print("orders:", orders_df.shape)
print("operations:", operations_df.shape)

machines_df.head(), orders_df.head(), operations_df.head()


machines: (8, 6)
shifts: (24, 4)
orders: (30, 7)
operations: (116, 10)


(   machine_id machine_group machine_name  daily_capacity_minutes  efficiency  \
 0    M_CUT_01           CUT     M CUT 01                     540       0.925   
 1    M_CUT_02           CUT     M CUT 02                     540       1.040   
 2   M_WELD_01          WELD    M WELD 01                     540       0.996   
 3   M_WELD_02          WELD    M WELD 02                     540       0.970   
 4  M_PAINT_01         PAINT   M PAINT 01                     540       0.881   
 
    can_preempt  
 0        False  
 1        False  
 2        False  
 3        False  
 4        False  ,
   order_id product_family  priority priority_label        release_time  \
 0  ORD_001        FRAME_A         3         normal 2026-04-20 08:10:00   
 1  ORD_002        FRAME_A         1       critical 2026-04-20 09:05:00   
 2  ORD_003        FRAME_A         3         normal 2026-04-20 09:10:00   
 3  ORD_004        FRAME_B         3         normal 2026-04-20 08:45:00   
 4  ORD_005          KIT_C  

## 3. Machine downtime scenarios

This is the main disruption scenario for the demo: a **short machine stoppage** that can trigger downstream delays.

A useful modeling choice is to store **both the estimated and the actual downtime**.  
That makes it possible to simulate a more realistic replanning workflow:

- operators initially know only the early estimate,
- the true duration becomes clearer over time,
- the scheduler or rescheduler can react as new information arrives.

In [5]:
def generate_downtime_scenarios(
    machines_df: pd.DataFrame,
    scenario_machine_group: str = "WELD",
    event_day: int = 0,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Create a small set of downtime scenarios for replanning experiments."""
    candidate_machines = machines_df.loc[
        machines_df["machine_group"] == scenario_machine_group, "machine_id"
    ].tolist()
    machine_id = candidate_machines[0]

    event_start = BASE_DATE + pd.Timedelta(days=event_day, hours=2, minutes=30)

    # The baseline row is useful when you want to compare a disrupted plan
    # against the original schedule built without any downtime.
    scenario_rows = [
        {
            "scenario_name": "baseline_no_disruption",
            "description": "Initial plan without machine stoppage",
            "machine_id": None,
            "event_start": pd.NaT,
            "estimated_duration_minutes": 0,
            "actual_duration_minutes": 0,
        },
        {
            "scenario_name": "optimistic_estimate",
            "description": "Stoppage initially estimated as 5 minutes",
            "machine_id": machine_id,
            "event_start": event_start,
            "estimated_duration_minutes": 5,
            "actual_duration_minutes": 20,
        },
        {
            "scenario_name": "pessimistic_estimate",
            "description": "Stoppage initially estimated as 20 minutes",
            "machine_id": machine_id,
            "event_start": event_start,
            "estimated_duration_minutes": 20,
            "actual_duration_minutes": 20,
        },
        {
            "scenario_name": "updated_after_10_min",
            "description": "Initial estimate 5 min, then updated to 20 min after 10 min",
            "machine_id": machine_id,
            "event_start": event_start,
            "estimated_duration_minutes": 5,
            "actual_duration_minutes": 20,
        },
    ]

    downtime_rows = []
    for i, sc in enumerate(scenario_rows[1:], start=1):
        downtime_rows.append(
            {
                "event_id": f"DT_{i:03d}",
                "machine_id": sc["machine_id"],
                "event_start": sc["event_start"],
                "estimated_duration_minutes": sc["estimated_duration_minutes"],
                "actual_duration_minutes": sc["actual_duration_minutes"],
                "reason": "unexpected_micro_stoppage",
                "scenario_name": sc["scenario_name"],
            }
        )

    return pd.DataFrame(downtime_rows), pd.DataFrame(scenario_rows)


downtime_df, scenarios_df = generate_downtime_scenarios(machines_df)
downtime_df

,event_id,machine_id,event_start,estimated_duration_minutes,actual_duration_minutes,reason,scenario_name
0,DT_001,M_WELD_01,2026-04-20 10:30:00,5,20,unexpected_micro_stoppage,optimistic_estimate
1,DT_002,M_WELD_01,2026-04-20 10:30:00,20,20,unexpected_micro_stoppage,pessimistic_estimate
2,DT_003,M_WELD_01,2026-04-20 10:30:00,5,20,unexpected_micro_stoppage,updated_after_10_min


## 4. Small embedded benchmark instance

Synthetic data is great for demos, but less convenient when you want a stable regression test.  
For that reason, the notebook also creates a small **embedded benchmark instance** inspired by the classic job-shop style.

It is meant to complement the synthetic dataset, not replace it:
- it is small,
- it runs quickly,
- and it is easy to use when debugging solver logic or objective functions.

In [6]:
def build_embedded_benchmark_instance() -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Build a tiny deterministic benchmark instance for quick solver checks."""
    # Each order is defined as a route of (machine_group, processing_time_minutes).
    instance = {
        "B_001": [("CUT", 10), ("WELD", 20), ("ASSY", 15)],
        "B_002": [("WELD", 15), ("CUT", 10), ("PACK", 10)],
        "B_003": [("ASSY", 20), ("PAINT", 15), ("PACK", 10)],
        "B_004": [("CUT", 15), ("ASSY", 10), ("PACK", 15)],
        "B_005": [("WELD", 10), ("PAINT", 20), ("ASSY", 10)],
        "B_006": [("CUT", 20), ("WELD", 10), ("PACK", 10)],
    }

    bench_orders = []
    bench_ops = []
    for i, (oid, routing) in enumerate(instance.items(), start=1):
        release = BASE_DATE
        nominal = sum(d for _, d in routing)
        deadline = release + pd.Timedelta(minutes=nominal + 90)

        bench_orders.append(
            {
                "order_id": oid,
                "product_family": "BENCHMARK",
                "priority": 3,
                "priority_label": "normal",
                "release_time": release,
                "deadline": deadline,
                "customer_segment": "benchmark",
            }
        )

        for seq, (group, dur) in enumerate(routing, start=1):
            bench_ops.append(
                {
                    "operation_id": f"{oid}_OP_{seq:02d}",
                    "order_id": oid,
                    "sequence_index": seq,
                    "machine_group_required": group,
                    "preferred_machine_id": MACHINE_GROUPS[group][0],
                    "processing_time_minutes": dur,
                    "setup_time_minutes": 0,
                    "release_time": release,
                    "deadline": deadline,
                    "is_outsourcable": False,
                }
            )

    bench_orders_df = pd.DataFrame(bench_orders)
    bench_ops_df = pd.DataFrame(bench_ops)
    bench_meta_df = pd.DataFrame(
        [
            {
                "instance_name": "embedded_benchmark_small",
                "num_orders": len(bench_orders_df),
                "num_operations": len(bench_ops_df),
            }
        ]
    )

    return bench_orders_df, bench_ops_df, bench_meta_df


bench_orders_df, bench_ops_df, bench_meta_df = build_embedded_benchmark_instance()
bench_orders_df.head(), bench_ops_df.head(), bench_meta_df

(  order_id product_family  priority priority_label        release_time  \
 0    B_001      BENCHMARK         3         normal 2026-04-20 08:00:00   
 1    B_002      BENCHMARK         3         normal 2026-04-20 08:00:00   
 2    B_003      BENCHMARK         3         normal 2026-04-20 08:00:00   
 3    B_004      BENCHMARK         3         normal 2026-04-20 08:00:00   
 4    B_005      BENCHMARK         3         normal 2026-04-20 08:00:00   
 
              deadline customer_segment  
 0 2026-04-20 10:15:00        benchmark  
 1 2026-04-20 10:05:00        benchmark  
 2 2026-04-20 10:15:00        benchmark  
 3 2026-04-20 10:10:00        benchmark  
 4 2026-04-20 10:10:00        benchmark  ,
   operation_id order_id  sequence_index machine_group_required  \
 0  B_001_OP_01    B_001               1                    CUT   
 1  B_001_OP_02    B_001               2                   WELD   
 2  B_001_OP_03    B_001               3                   ASSY   
 3  B_002_OP_01    B_002   

## 5. Useful derived features

For scheduling experiments and downstream analytics, it is often helpful to precompute a few order-level features:

- total operation time,
- number of operations,
- number of machine groups touched by the order,
- available slack,
- deadline tightness.

In [7]:
def build_order_features(operations_df: pd.DataFrame, orders_df: pd.DataFrame) -> pd.DataFrame:
    """Aggregate operation-level data into a few useful order-level features."""
    agg = (
        operations_df.assign(
            total_op_time=lambda x: x["processing_time_minutes"] + x["setup_time_minutes"]
        )
        .groupby("order_id", as_index=False)
        .agg(
            total_operation_minutes=("total_op_time", "sum"),
            num_operations=("operation_id", "count"),
            num_machine_groups=("machine_group_required", "nunique"),
        )
    )

    out = orders_df.merge(agg, on="order_id", how="left")
    out["available_slack_minutes"] = (
        (pd.to_datetime(out["deadline"]) - pd.to_datetime(out["release_time"])).dt.total_seconds() / 60
        - out["total_operation_minutes"]
    )
    out["deadline_tightness_ratio"] = out["available_slack_minutes"] / out["total_operation_minutes"]
    return out.sort_values(["priority", "deadline"])


order_features_df = build_order_features(operations_df, orders_df)
order_features_df.head()

,order_id,product_family,priority,priority_label,release_time,deadline,customer_segment,total_operation_minutes,num_operations,num_machine_groups,available_slack_minutes,deadline_tightness_ratio
20,ORD_021,KIT_C,1,critical,2026-04-20 09:25:00,2026-04-20 18:00:00,Distributor,195,3,3,320.0,1.641026
22,ORD_023,FRAME_B,1,critical,2026-04-20 08:15:00,2026-04-20 18:15:00,OEM,290,4,4,310.0,1.068966
17,ORD_018,MODULE_D,1,critical,2026-04-20 09:15:00,2026-04-20 18:39:00,Rush,295,3,3,269.0,0.911864
1,ORD_002,FRAME_A,1,critical,2026-04-20 09:05:00,2026-04-20 21:24:00,Service,445,5,5,294.0,0.660674
21,ORD_022,FRAME_A,1,critical,2026-04-20 08:15:00,2026-04-20 21:45:00,Rush,405,5,5,405.0,1.000000


## 6. Consistency checks

Before exporting the dataset, we run a few lightweight validation checks to catch structural issues early.

In [8]:
def validate_dataset(
    machines_df: pd.DataFrame,
    shifts_df: pd.DataFrame,
    orders_df: pd.DataFrame,
    operations_df: pd.DataFrame,
) -> List[str]:
    """Run a small set of structural checks before export."""
    errors = []

    if machines_df["machine_id"].duplicated().any():
        errors.append("Duplicate machine_id in machines_df")

    if orders_df["order_id"].duplicated().any():
        errors.append("Duplicate order_id in orders_df")

    if operations_df["operation_id"].duplicated().any():
        errors.append("Duplicate operation_id in operations_df")

    missing_orders = set(operations_df["order_id"]) - set(orders_df["order_id"])
    if missing_orders:
        errors.append(f"Operations reference missing orders: {sorted(missing_orders)}")

    known_groups = set(machines_df["machine_group"])
    bad_groups = set(operations_df["machine_group_required"]) - known_groups
    if bad_groups:
        errors.append(f"Unknown machine groups in operations: {sorted(bad_groups)}")

    if (operations_df["processing_time_minutes"] <= 0).any():
        errors.append("Found non-positive processing times")

    if (pd.to_datetime(orders_df["deadline"]) < pd.to_datetime(orders_df["release_time"])).any():
        errors.append("Some orders have a deadline earlier than release_time")

    return errors


validation_errors = validate_dataset(machines_df, shifts_df, orders_df, operations_df)
print("validation_errors:", validation_errors if validation_errors else "none")

validation_errors: none


## 7. Export to CSV

At this point, the generated data can already serve as input for:
- an OR-Tools CP-SAT model,
- a heuristic scheduler,
- a QUBO transformation pipeline,
- or a simulation / dashboard layer such as Streamlit.

In [9]:
def save_dataset_bundle(
    output_dir: Path,
    machines_df: pd.DataFrame,
    shifts_df: pd.DataFrame,
    orders_df: pd.DataFrame,
    operations_df: pd.DataFrame,
    downtime_df: pd.DataFrame,
    scenarios_df: pd.DataFrame,
    prefix: str = "synthetic_demo",
) -> Dict[str, Path]:
    """Write one dataset bundle to disk and return the saved paths."""
    bundle_dir = output_dir / prefix
    bundle_dir.mkdir(parents=True, exist_ok=True)

    paths = {
        "machines": bundle_dir / "machines.csv",
        "shifts": bundle_dir / "shifts.csv",
        "orders": bundle_dir / "orders.csv",
        "operations": bundle_dir / "operations.csv",
        "downtime_events": bundle_dir / "downtime_events.csv",
        "scenarios": bundle_dir / "scenarios.csv",
    }

    machines_df.to_csv(paths["machines"], index=False)
    shifts_df.to_csv(paths["shifts"], index=False)
    orders_df.to_csv(paths["orders"], index=False)
    operations_df.to_csv(paths["operations"], index=False)
    downtime_df.to_csv(paths["downtime_events"], index=False)
    scenarios_df.to_csv(paths["scenarios"], index=False)

    return paths


synthetic_paths = save_dataset_bundle(
    OUTPUT_DIR,
    machines_df,
    shifts_df,
    orders_df,
    operations_df,
    downtime_df,
    scenarios_df,
    prefix="synthetic_demo",
)

synthetic_paths

{'machines': PosixPath('generated_factory_demo_data/synthetic_demo/machines.csv'),
 'shifts': PosixPath('generated_factory_demo_data/synthetic_demo/shifts.csv'),
 'orders': PosixPath('generated_factory_demo_data/synthetic_demo/orders.csv'),
 'operations': PosixPath('generated_factory_demo_data/synthetic_demo/operations.csv'),
 'downtime_events': PosixPath('generated_factory_demo_data/synthetic_demo/downtime_events.csv'),
 'scenarios': PosixPath('generated_factory_demo_data/synthetic_demo/scenarios.csv')}

In [10]:
benchmark_paths = save_dataset_bundle(
    OUTPUT_DIR,
    machines_df,
    shifts_df,
    bench_orders_df,
    bench_ops_df,
    downtime_df,
    scenarios_df,
    prefix="embedded_benchmark",
)

bench_meta_df.to_csv(OUTPUT_DIR / "embedded_benchmark" / "benchmark_meta.csv", index=False)
benchmark_paths


{'machines': PosixPath('generated_factory_demo_data/embedded_benchmark/machines.csv'),
 'shifts': PosixPath('generated_factory_demo_data/embedded_benchmark/shifts.csv'),
 'orders': PosixPath('generated_factory_demo_data/embedded_benchmark/orders.csv'),
 'operations': PosixPath('generated_factory_demo_data/embedded_benchmark/operations.csv'),
 'downtime_events': PosixPath('generated_factory_demo_data/embedded_benchmark/downtime_events.csv'),
 'scenarios': PosixPath('generated_factory_demo_data/embedded_benchmark/scenarios.csv')}

## 8. Quick data preview

This is a simple sanity check before plugging the generated data into a solver.

In [11]:
print("Synthetic orders by priority")
display(orders_df["priority_label"].value_counts().rename_axis("priority").reset_index(name="count"))

print("\nOperations per machine group")
display(operations_df["machine_group_required"].value_counts().rename_axis("machine_group").reset_index(name="count"))

print("\nDowntime scenarios")
display(downtime_df)


Synthetic orders by priority


,priority,count
0,normal,15
1,low,6
2,critical,5
3,high,4



Operations per machine group


,machine_group,count
0,ASSY,30
1,CUT,26
2,PACK,26
3,WELD,22
4,PAINT,12



Downtime scenarios


,event_id,machine_id,event_start,estimated_duration_minutes,actual_duration_minutes,reason,scenario_name
0,DT_001,M_WELD_01,2026-04-20 10:30:00,5,20,unexpected_micro_stoppage,optimistic_estimate
1,DT_002,M_WELD_01,2026-04-20 10:30:00,20,20,unexpected_micro_stoppage,pessimistic_estimate
2,DT_003,M_WELD_01,2026-04-20 10:30:00,5,20,unexpected_micro_stoppage,updated_after_10_min


## 9. Minimal scaffold for solver input

The helper below loads a saved dataset bundle into a dictionary that is convenient for a downstream model builder.

In [12]:
def load_bundle_for_scheduler(bundle_dir: Path) -> Dict[str, pd.DataFrame]:
    """Load a saved dataset bundle into memory for downstream scheduling code."""
    return {
        # machines.csv contains master data only, so no date parsing is needed here.
        "machines": pd.read_csv(bundle_dir / "machines.csv") if (bundle_dir / "machines.csv").exists() else None,
        "shifts": pd.read_csv(bundle_dir / "shifts.csv", parse_dates=["shift_start", "shift_end"]),
        "orders": pd.read_csv(bundle_dir / "orders.csv", parse_dates=["release_time", "deadline"]),
        "operations": pd.read_csv(bundle_dir / "operations.csv", parse_dates=["release_time", "deadline"]),
        "downtime_events": pd.read_csv(bundle_dir / "downtime_events.csv", parse_dates=["event_start"]),
        "scenarios": pd.read_csv(bundle_dir / "scenarios.csv", parse_dates=["event_start"]),
    }

# A natural next step would be to pass this dictionary into your model builder,
# for example a CP-SAT, MILP, heuristic, or QUBO conversion pipeline.